# Deep Learning for NLP – Lab Exercise 1
## Biomedical Text Classification with PyTorch

**Student name:** 

**Date:**

---

### Overview
This notebook implements and compares two neural architectures for **multi-label** classification of biomedical articles from the **LitCovid** dataset (BioCreative VII) into seven topics: *Treatment, Diagnosis, Prevention, Mechanism, Transmission, Epidemic Forecasting, Case Report*.

1. A **Bag-of-Words (BOW)** classifier based on word embeddings.
2. A **Convolutional Neural Network (CNN)** for text classification.

### Contents
0. Setup and reproducibility
1. Dataset
2. Text preprocessing
3. PyTorch tensors
4. Neural network definition
5. Loss function
6. Training loop
7. Evaluation
8. Experiments and model comparison
9. Discussion
10. Optional extensions

## 0. Setup and reproducibility
*Describe your environment (CPU/GPU, library versions) and the global hyperparameters you chose.*

In [ ]:
!pip install datasets scikit-learn

In [ ]:
import re, time, random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch as th
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as ag
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# TODO
# Fix random seeds (random, numpy, torch) for reproducibility.
SEED = 42

# TODO
# Select the device (CPU / GPU).

# TODO
# Define global hyperparameters (MAX_LEN, BATCH_SIZE, EPOCHS, learning rate...).

## 1. Dataset
### 1.1 Load the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("KushT/LitCovid_BioCreative")

print(dataset)

In [ ]:
print(dataset["train"][0])

*Note on the splits: check the size of each split and explain which one you use as the development set (the official development set has 6,239 articles).*

In [ ]:
# TODO
# Select the training and development splits.

### Questions 1.1

**1. How many articles are available in each dataset split?**

*Answer:* 

**2. What information is contained in the `text` field?**

*Answer:* 

**3. What is the representation of the labels?**

*Answer:* 

**4. Can an article have multiple labels?**

*Answer:* 

**5. What is the distribution of the seven topics?**

*Answer:* 

### 1.2 Explore the label distribution

In [ ]:
labels = [
    "Treatment",
    "Diagnosis",
    "Prevention",
    "Mechanism",
    "Transmission",
    "Epidemic Forecasting",
    "Case Report"
]

In [ ]:
# TODO
# Calculate the number of positive examples for each label.

In [ ]:
# TODO
# Visualize the distribution using a bar plot.

In [ ]:
# TODO (optional)
# Number of labels per article.

### Questions 1.2

**1. Which topic is the most frequent?**

*Answer:* 

**2. Which topic is the least frequent?**

*Answer:* 

**3. Is the dataset balanced?**

*Answer:* 

**4. What problems might class imbalance create during training?**

*Answer:* 

## 2. Text preprocessing

In [ ]:
def clean_str(string, tolower=True):
    """
    Basic text cleaning and tokenization.
    """
    string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
    string = re.sub(r"\s{2,}", " ", string)

    if tolower:
        string = string.lower()

    return string.strip()


def tokenize(text):
    return clean_str(text).split()

In [ ]:
# TODO
# Create tokenized versions of the training and development articles.

*Implementation choices: explain any decision you made during preprocessing (e.g. which field you use, how the `[Title]`, `[Abstract]`, `[Keywords]` markers are handled).*

### 2.1 Vocabulary construction

In [ ]:
word2id = {
    "<PAD>": 0,
    "<UNK>": 1
}

# TODO
# Construct the vocabulary using the training data only.

In [ ]:
# TODO (optional)
# Vocabulary size, most frequent words, OOV rate on the development set.

### Question 2.1

**1. Why should we avoid using the development data when constructing the vocabulary?**

*Answer:* 

### 2.2 Convert text into numerical representations

In [ ]:
def text_to_ids(tokens, word2id):
    # TODO
    pass

In [ ]:
# TODO
# Test the function and convert the training and development articles.

### 2.3 Padding and batching
*Justify your choice of `MAX_LEN` (e.g. with the distribution of article lengths).*

In [ ]:
# TODO (optional)
# Analyse the distribution of article lengths.

In [ ]:
MAX_LEN = 500

# TODO
# Implement padding and truncation.

### Questions 2.3

**1. What happens when an article is longer than `MAX_LEN`?**

*Answer:* 

**2. Why do we need padding?**

*Answer:* 

**3. What is the difference between `<PAD>` and `<UNK>`?**

*Answer:* 

**4. What are the potential disadvantages of using a fixed maximum length?**

*Answer:* 

## 3. Convert the data into PyTorch tensors
Requirements:
- Input sequences → `torch.long`
- Labels → `torch.float` (required by `BCEWithLogitsLoss`)
- No gradient required for the input tensors

In [ ]:
# TODO
# Convert the training and development datasets into tensors.

In [ ]:
# TODO
# Create the DataLoaders (mini-batches).

*Explain your choices (batch size, shuffling, dtypes).*

## 4. Neural network definition
### 4.1 Bag-of-Words classifier
*Describe your architecture: embedding layer, aggregation (mean pooling), hidden layer, output layer.*

In [ ]:
class CBOW_classifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim=128,
                 num_labels=7, padding_idx=0):

        super(CBOW_classifier, self).__init__()

        # TODO
        # Define the embedding layer.

        # TODO
        # Define an optional hidden layer.

        # TODO
        # Define the output layer.

    def forward(self, inputs):

        # TODO
        # Retrieve word embeddings.

        # TODO
        # Compute the mean of the embeddings.

        # TODO
        # Apply the hidden layer.

        # TODO
        # Produce seven output logits.

        return outputs

In [ ]:
# TODO
# Sanity check: instantiate the model and check the output shape on one batch.

### Questions 4.1

**1. Why do we use embeddings instead of one-hot representations?**

*Answer:* 

**2. What is the difference between sum pooling and mean pooling?**

*Answer:* 

**3. Why do we need seven output neurons?**

*Answer:* 

**4. Why should we not apply a softmax function to the output?**

*Answer:* 

**5. What happens if we use a sigmoid function instead?**

*Answer:* 

### 4.2 Convolutional Neural Network (CNN)
*Describe your architecture: embedding layer, convolutional layers and window sizes, padding, activation, max pooling, output layer.*

In [ ]:
class CNN_classifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim,
                 num_filters=100, kernel_sizes=[2, 3, 5],
                 num_labels=7, padding_idx=0):

        super(CNN_classifier, self).__init__()

        # TODO
        # Define the embedding layer.

        # TODO
        # Define convolutional layers.

        # TODO
        # Define the output layer.

    def forward(self, inputs):

        # TODO
        # Retrieve word embeddings.

        # TODO
        # Apply convolutional layers.

        # TODO
        # Apply a non-linear activation function.

        # TODO
        # Apply max pooling.

        # TODO
        # Concatenate the representations.

        # TODO
        # Produce seven output logits.

        return outputs

In [ ]:
# TODO
# Sanity check: output shape on one batch and on a very short sequence.

### Questions 4.2

**1. What is the role of convolutional filters?**

*Answer:* 

**2. Why do we use multiple window sizes?**

*Answer:* 

**3. What is the purpose of max pooling?**

*Answer:* 

**4. How does the CNN differ from the BOW classifier?**

*Answer:* 

**5. What types of linguistic patterns might a CNN learn from biomedical articles?**

*Answer:* 

## 5. Loss function

In [ ]:
criterion = nn.BCEWithLogitsLoss()

### Questions 5

**1. What is the difference between `BCELoss` and `BCEWithLogitsLoss`?**

*Answer:* 

**2. Why is `BCEWithLogitsLoss` appropriate for this task?**

*Answer:* 

**3. What is the role of the sigmoid function?**

*Answer:* 

**4. How would class imbalance affect the loss function?**

*Answer:* 

**Optional – `pos_weight`:** Could it help improve performance on underrepresented topics?

*Answer:*

In [ ]:
# TODO (optional)
# Compute pos_weight = (#negatives / #positives) for each label.

## 6. Training loop
*The training loop includes: parameterizable number of epochs, mini-batch training, forward propagation, loss computation, backpropagation, parameter updates and validation after each epoch. Gradient computation is disabled during validation.*

In [ ]:
def train_model(model, train_loader, dev_loader,
                optimizer, criterion, epochs):

    for epoch in range(epochs):

        model.train()

        # TODO
        # Implement the training loop.

        model.eval()

        # TODO
        # Evaluate the model on the development set.

        print(
            f"Epoch {epoch+1}: "
            f"Train Loss = ..., "
            f"Dev Loss = ..."
        )

In [ ]:
# TODO
# Instantiate a model and the optimizer, then train it.
# optimizer = th.optim.Adam(model.parameters(), lr=0.001)

## 7. Evaluation
Metrics: precision, recall, micro-F1, macro-F1. Predictions are obtained by applying a sigmoid to the outputs and a threshold of 0.5.

In [ ]:
# TODO
# Get the model outputs on the development set.

# probabilities = th.sigmoid(outputs)
# predictions = (probabilities >= 0.5).int()

In [ ]:
# TODO
# Calculate precision, recall, micro-F1 and macro-F1.

In [ ]:
# TODO
# Per-label results (classification_report).

In [ ]:
# TODO
# Compare thresholds 0.5 and 0.3.

### Questions 7

**1. What is the difference between micro-F1 and macro-F1?**

*Answer:* 

**2. Why is macro-F1 particularly relevant for this dataset?**

*Answer:* 

**3. What happens if we change the prediction threshold from 0.5 to 0.3?**

*Answer:* 

**4. Can a model have good micro-F1 but poor macro-F1?**

*Answer:* 

**5. Which metric would you prioritize for biomedical literature classification? Explain your choice.**

*Answer:* 

## 8. Experiments and model comparison

| Experiment | Architecture | Embedding dimension | Hidden dimension |
|---|---|---|---|
| 1 | BOW | 50  | 128 |
| 2 | BOW | 100 | 128 |
| 3 | CNN | 100 | 128 |
| ... | *(additional configurations)* | | |

For each experiment record: training loss, development loss, micro-F1, macro-F1 and training time.

In [ ]:
# TODO
# Helper function to run one experiment and return its metrics + training time.

### Experiment 1 – BOW, embedding 50, hidden 128

In [ ]:
# TODO

### Experiment 2 – BOW, embedding 100, hidden 128

In [ ]:
# TODO

### Experiment 3 – CNN, embedding 100, hidden 128

In [ ]:
# TODO

### Additional experiments

In [ ]:
# TODO (optional)

### Results table

In [ ]:
# TODO
# Summarise all experiments in a table (pandas DataFrame).

In [ ]:
# TODO
# Plot training / development loss curves.

In [ ]:
# TODO
# Per-label F1 for each experiment.

### Discussion of the experiments

**1. Which architecture performs best?**

*Answer:* 

**2. Does increasing the embedding dimension improve performance?**

*Answer:* 

**3. Does the CNN outperform the BOW model?**

*Answer:* 

**4. Which topics are the most difficult to predict?**

*Answer:* 

**5. What are the limitations of the implemented architectures?**

*Answer:* 

### 9. Discussion

**1. Why is biomedical text classification an important NLP application?**

*Answer:* 

**2. What are the main challenges of classifying scientific articles?**

*Answer:* 

**3. Why is multi-label classification more challenging than binary classification?**

*Answer:* 

**4. How could pretrained biomedical word embeddings improve performance?**

*Answer:* 

**5. How might a transformer-based architecture improve the results?**

*Answer:* 

## 10. Optional extensions
### Extension 1 – Class imbalance
*Does weighted binary cross-entropy (`pos_weight`) improve performance on less frequent topics?*

In [ ]:
# TODO (optional)

*Discussion:*

### Extension 2 – Threshold optimization
*Select a different threshold for each label using only the development set.*

In [ ]:
# TODO (optional)

*Discussion:*

### Extension 3 – Pretrained embeddings
*Replace randomly initialized embeddings with pretrained biomedical word embeddings.*

In [ ]:
# TODO (optional)

*Discussion:*

### Extension 4 – Transformer models
*Compare your best model with a pretrained biomedical language model (e.g. BioBERT).*

In [ ]:
# TODO (optional)

*Discussion of advantages and limitations compared with BOW and CNN:*

## Conclusions
*Summarise your main findings: best model, effect of the hyperparameters, hardest topics, and possible improvements.*